# FrankenSim Cooling 0.1 Reference Workflow

This notebook demonstrates the policy-gated Python workflow for **FrankenSim**.
Under Decision Record **ADPT-2026-07**, the Python SDK operates strictly out-of-process via structured JSON lines against the certified Rust binary. All error bounds, evidence colors, Five Explicits, and certificates are preserved bit-accurately.

In [1]:
import os
import sys
from pathlib import Path
import tempfile

# Resolve repository root
p = Path.cwd().resolve()
while p != p.parent and not (p / "Cargo.toml").exists():
    p = p.parent
repo_root = p

sys.path.insert(0, str(repo_root / "python"))

from frankensim import FrankenSimClient, ExitCode

client = FrankenSimClient()
print(f"Using FrankenSim binary at: {client.binary_path}")

## 1. Structural Project Validation
We validate the heatsink-fan simulation project against its schema and semantic rules.

In [2]:
project_path = repo_root / "examples" / "heatsink-fan" / "heatsink-fan.fsim"
val_result = client.validate(project_path, strict=True)

print(f"Validation Status: {val_result.status}")
print(f"Project Hash:      {val_result.project_hash}")
print(f"Finding Count:     {val_result.finding_count}")
print(f"Authority:         {val_result.authority}")

## 2. Geometry Import into Durable Ledger
Import the STL geometry into an SQLite ledger, verifying watertightness and boundary conditions.

In [3]:
stl_path = repo_root / "examples" / "heatsink-fan" / "heatsink.stl"
ledger_dir = tempfile.mkdtemp(prefix="frankensim_nb_")
ledger_path = Path(ledger_dir) / "cooling_run.db"

imp_result = client.import_mesh(
    project_path=project_path,
    source_path=stl_path,
    ledger_path=ledger_path,
    unit="m",
    max_hole_edges=0,
    strict=True,
)

print(f"Import Status:  {imp_result.status}")
print(f"Artifact Count: {imp_result.artifact_count}")
print(f"Op ID:          {imp_result.op_id}")
print(f"Project Hash:   {imp_result.project_hash}")

## 3. Solve Orchestration with Declared Material Cards
Execute the solve stages (Airflow Network, Conduction, Boundary Coupling) under explicit budgets.

In [4]:
mat_pack = repo_root / "data" / "reference-project" / "aa6061.fsmcdpk"

solve_outcome = client.solve(
    project_path=project_path,
    ledger_path=ledger_path,
    materials=[mat_pack],
    strict=False,  # Stage gaps are reported honestly as exit::UNAVAILABLE
)

print(f"Solve Status:       {solve_outcome.status}")
print(f"Exit Code:          {solve_outcome.exit_code}")
print(f"Stages Completed:   {solve_outcome.stages_completed}")
for diag in solve_outcome.diagnostics:
    print(f"  [{diag.severity}] {diag.code}: {diag.message}")

## 4. Engineering Report & Evidence Package Generation
Generate deterministic HTML and JSON twin reports and verify offline package integrity.

In [5]:
run_id = "feedface000000000000000000000000feedface000000000000000000000000"
report_res = client.report(run_id=run_id, ledger_path=ledger_path, strict=False)
print(f"Report HTML: {report_res.html_path}")
print(f"Report JSON: {report_res.json_path}")

pkg_res = client.package(run_id=run_id, ledger_path=ledger_path, strict=False)
print(f"Package Archive: {pkg_res.package_path}")
print(f"Audit Verdict:   {pkg_res.audit_verdict}")